In [ ]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("input.xlsx")

df["Plan"] = pd.to_numeric(df["Plan"], errors="coerce").fillna(0)
df["Cycle Time"] = pd.to_numeric(df["Cycle Time"], errors="coerce").fillna(0)

# =============================
# Constants
# =============================
USABLE_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# Tracking structures
# =============================
machine_load = {m: 0 for m in USABLE_MACHINES}
machine_plan = defaultdict(list)
rejection_log = []

# =============================
# Allocation logic
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    required_qty = row["Plan"]
    cycle_time = row["Cycle Time"]

    if required_qty <= 0 or cycle_time <= 0:
        continue

    required_time = required_qty * cycle_time

    vertical_machines = str(row["Vertical Machines"]).split(",")
    vertical_machines = [m.strip() for m in vertical_machines]

    eligible = [m for m in vertical_machines if m in USABLE_MACHINES]

    if not eligible:
        rejection_log.append((child, "No eligible 120T machine"))
        continue

    remaining_time = required_time

    eligible.sort(key=lambda m: machine_load[m])

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejection_log.append((
            child,
            f"Shortfall qty {round(remaining_time / cycle_time, 2)}"
        ))

# =============================
# DISPLAY RESULTS
# =============================

print("\n================ MACHINE-WISE PLAN ================\n")
for m, plans in machine_plan.items():
    print(f"🔧 {m}")
    display(pd.DataFrame(plans))
    print("-" * 60)

print("\n================ MACHINE LOAD SUMMARY ================\n")
load_df = pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in USABLE_MACHINES
])
display(load_df)

print("\n================ REJECTIONS / SHORTFALLS ================\n")
if rejection_log:
    display(pd.DataFrame(rejection_log, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections. All plans feasible.")


In [ ]:
import pandas as pd
from collections import defaultdict

# =============================
# Load data
# =============================
df = pd.read_excel("input.xlsx")

df["Plan"] = pd.to_numeric(df["Plan"], errors="coerce").fillna(0)
df["Cycle Time"] = pd.to_numeric(df["Cycle Time"], errors="coerce").fillna(0)

# =============================
# EXPLICIT MACHINE ALLOW LIST
# =============================
ALLOWED_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

MACHINE_CAPACITY = 3960  # minutes (22 hrs × 3 days)

# =============================
# Tracking
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejection_log = []

# =============================
# Allocation
# =============================
for _, row in df.iterrows():

    child = row["Child Part"]
    required_qty = row["Plan"]
    cycle_time = row["Cycle Time"]

    if required_qty <= 0 or cycle_time <= 0:
        continue

    required_time = required_qty * cycle_time

    vertical_machines = [
        m.strip() for m in str(row["Vertical Machines"]).split(",")
    ]

    # STRICT FILTER — name-based only
    eligible = [m for m in vertical_machines if m in ALLOWED_MACHINES]

    if not eligible:
        rejection_log.append((child, "No allowed machine"))
        continue

    remaining_time = required_time

    # Load balancing
    eligible.sort(key=lambda m: machine_load[m])

    for m in eligible:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / cycle_time

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejection_log.append((
            child,
            f"Shortfall qty {round(remaining_time / cycle_time, 2)}"
        ))

# =============================
# DISPLAY RESULTS
# =============================

print("\n========== MACHINE-WISE PLAN ==========\n")
for m in sorted(ALLOWED_MACHINES):
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No assigned parts")
    print("-" * 50)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in sorted(ALLOWED_MACHINES)
]))

print("\n========== REJECTIONS ==========\n")
if rejection_log:
    display(pd.DataFrame(rejection_log, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")


In [ ]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE
# ────────────────────────────────────────────────
FILE_PATH = r"C:\Users\YourName\Documents\production_data.xlsx"   # ← your file path
SHEET_NAME = "Sheet1"                                             # ← change if needed
OUTPUT_FILE = "production_plan_120ton.xlsx"                       # where to save result

# Machine group (excluding fixed-tool machines MP-11, MP-15)
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Productive seconds per day per machine (8 hours = 8*60*60)
SECONDS_PER_DAY = 8 * 3600
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS

# Column names in your file (change only if they are different)
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_DAILY_PLAN  = "Daily Plan (of Switch Part Number)"
COL_MONTHLY_REQ = "Monthly Requirement (Child part)"
COL_MIN_QTY     = "Minimum Quantity (Child Part)"
COL_PLAN_QTY    = "Plan(Child Part calculated as Monthly Requirement / 31 * 3)"
COL_CYCLE_TIME  = "Cycle Time(of child part)"
COL_MACHINE     = "Vertical Machines"
# ────────────────────────────────────────────────

def load_data():
    if FILE_PATH.lower().endswith('.csv'):
        df = pd.read_csv(FILE_PATH)
    else:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    
    # Clean column names (remove extra spaces)
    df.columns = df.columns.str.strip()
    
    # Keep only relevant columns
    keep_cols = [COL_CHILD, COL_SWITCH, COL_DAILY_PLAN, COL_MONTHLY_REQ,
                 COL_MIN_QTY, COL_PLAN_QTY, COL_CYCLE_TIME, COL_MACHINE]
    df = df[keep_cols].copy()
    
    # Convert numeric columns
    for col in [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MONTHLY_REQ, COL_MIN_QTY, COL_DAILY_PLAN]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.dropna(subset=[COL_PLAN_QTY, COL_CYCLE_TIME])  # must have qty & time
    
    return df


def calculate_loads(df):
    # Calculate production seconds needed
    df['Production_Seconds'] = df[COL_PLAN_QTY] * df[COL_CYCLE_TIME]
    df['Production_Hours']   = df['Production_Seconds'] / 3600
    
    # Normalize machine names
    df[COL_MACHINE] = df[COL_MACHINE].str.strip().str.upper()
    
    # Split into fixed and flexible
    fixed = df[df[COL_MACHINE].isin(ALLOWED_MACHINES)].copy()
    flexible = df[~df[COL_MACHINE].isin(ALLOWED_MACHINES) & 
                  (df[COL_MACHINE].str.lower() != 'fixed') & 
                  df[COL_MACHINE].notna()].copy()  # assume others are flexible
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    # Greedy load balancing: assign to machine with current lowest load
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}        # in seconds
    assignments = defaultdict(list)
    
    # Sort by biggest jobs first (helps balance better)
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        # Find machine with lowest current load
        best_machine = min(machine_load, key=machine_load.get)
        assignments[best_machine].append(row)
        machine_load[best_machine] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    # Fixed assignments
    for _, row in fixed.iterrows():
        m = row[COL_MACHINE]
        plan_rows.append({
            'Machine': m,
            'Child Part': row[COL_CHILD],
            'Switch Part': row[COL_SWITCH],
            'Plan Qty': row[COL_PLAN_QTY],
            'Cycle Time (s)': row[COL_CYCLE_TIME],
            'Total Seconds': row['Production_Seconds'],
            'Total Hours': row['Production_Hours'],
            'Type': 'Fixed'
        })
    
    # Flexible assignments
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Part': part[COL_SWITCH],
                'Plan Qty': part[COL_PLAN_QTY],
                'Cycle Time (s)': part[COL_CYCLE_TIME],
                'Total Seconds': part['Production_Seconds'],
                'Total Hours': part['Production_Hours'],
                'Type': 'Flexible'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Summary per machine
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Utilization %'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Available Hours'] = AVAILABLE_SECONDS_PER_MACHINE / 3600
    summary = summary[['Part Count', 'Total Hours', 'Available Hours', 'Utilization %']]
    
    return plan_df, summary


def main():
    print("Loading data...")
    df = load_data()
    
    print(f"Total Child Parts found: {len(df)}")
    
    fixed, flexible = calculate_loads(df)
    print(f"Fixed parts  : {len(fixed)}")
    print(f"Flexible parts: {len(flexible)}")
    
    flexible_assignments, machine_load_sec = assign_flexible_parts(flexible)
    
    plan_df, summary = create_final_plan(fixed, flexible_assignments)
    
    # Save to Excel
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_df.to_excel(writer, sheet_name='Detailed Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
    
    print("\n" + "="*60)
    print("           PRODUCTION PLAN CREATED")
    print("="*60)
    print("\nMachine Summary (3-day plan):")
    print(summary.round(2))
    
    print(f"\nDetailed plan saved to: {OUTPUT_FILE}")
    print("\nColumns in detailed sheet:")
    print(plan_df.columns.tolist())
    
    # Quick check for overload
    overloaded = summary[summary['Utilization %'] > 100]
    if not overloaded.empty:
        print("\nWARNING: Overloaded machines!")
        print(overloaded)


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE
# ────────────────────────────────────────────────
FILE_PATH = r"C:\Users\YourName\Documents\production_data.xlsx"   # ← CHANGE THIS to your actual file path
SHEET_NAME = "Sheet1"                                             # ← change if your sheet has a different name
OUTPUT_FILE = "production_plan_120ton.xlsx"                       # output file name

# Machines in 120 tonnage group (excluding MP-11 and MP-15)
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Productive seconds per day per machine (8 hours/day)
SECONDS_PER_DAY = 8 * 3600
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS

# Column names in your Excel/CSV file (adjust only if names are different)
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_DAILY_PLAN  = "Daily Plan (of Switch Part Number)"
COL_MONTHLY_REQ = "Monthly Requirement (Child part)"
COL_MIN_QTY     = "Minimum Quantity (Child Part)"
COL_PLAN_QTY    = "Plan(Child Part calculated as Monthly Requirement / 31 * 3)"
COL_CYCLE_TIME  = "Cycle Time(of child part)"
COL_MACHINE     = "Vertical Machines"
# ────────────────────────────────────────────────

def load_and_aggregate_data():
    # Read file
    if FILE_PATH.lower().endswith('.csv'):
        df = pd.read_csv(FILE_PATH)
    else:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    
    # Clean column names (remove extra spaces)
    df.columns = df.columns.str.strip()
    
    # Keep only needed columns
    keep_cols = [COL_CHILD, COL_SWITCH, COL_DAILY_PLAN, COL_MONTHLY_REQ,
                 COL_MIN_QTY, COL_PLAN_QTY, COL_CYCLE_TIME, COL_MACHINE]
    df = df[keep_cols].copy()
    
    # Convert numeric columns
    numeric_cols = [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MONTHLY_REQ, COL_MIN_QTY, COL_DAILY_PLAN]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop rows missing critical data
    df = df.dropna(subset=[COL_CHILD, COL_PLAN_QTY, COL_CYCLE_TIME])
    
    # ──── IMPORTANT: Take FIRST occurrence of each Child Part ─────
    # Preserve original file order and take first row per unique Child Part
    df = df.sort_index()  # ensure original order
    aggregated = df.groupby(COL_CHILD, as_index=False).first()
    
    # Add count of how many times each part appeared (for your info)
    counts = df[COL_CHILD].value_counts().reset_index(name='Appearance_Count')
    aggregated = aggregated.merge(counts, on=COL_CHILD, how='left')
    
    # Rename columns for clarity
    aggregated = aggregated.rename(columns={
        COL_PLAN_QTY:    'Plan_Qty',
        COL_CYCLE_TIME:  'Cycle_Time_s',
        COL_MACHINE:     'Assigned_Machine',
        COL_SWITCH:      'Switch_Part_Reference',
        COL_MONTHLY_REQ: 'Monthly_Req',
        COL_MIN_QTY:     'Min_Qty'
    })
    
    print(f"Original rows: {len(df)}")
    print(f"Unique Child Parts (using first occurrence): {len(aggregated)}")
    print(f"Parts that appeared multiple times: {len(aggregated[aggregated['Appearance_Count'] > 1])}")
    
    return aggregated


def calculate_loads(agg_df):
    agg_df['Production_Seconds'] = agg_df['Plan_Qty'] * agg_df['Cycle_Time_s']
    agg_df['Production_Hours']   = agg_df['Production_Seconds'] / 3600
    
    # Normalize machine names
    agg_df['Assigned_Machine'] = agg_df['Assigned_Machine'].astype(str).str.strip().str.upper()
    
    # Split into fixed and flexible
    fixed_mask = agg_df['Assigned_Machine'].isin(ALLOWED_MACHINES)
    fixed = agg_df[fixed_mask].copy()
    flexible = agg_df[~fixed_mask].copy()
    
    print(f"Fixed parts   : {len(fixed)}")
    print(f"Flexible parts: {len(flexible)}")
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    # Greedy balancing: assign biggest jobs first to least loaded machine
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}  # current load in seconds
    assignments = defaultdict(list)
    
    # Sort descending by production time (helps better balance)
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        best_machine = min(machine_load, key=machine_load.get)
        assignments[best_machine].append(row)
        machine_load[best_machine] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    # Fixed parts
    for _, row in fixed.iterrows():
        plan_rows.append({
            'Machine': row['Assigned_Machine'],
            'Child Part': row[COL_CHILD],
            'Switch Reference': row['Switch_Part_Reference'],
            'Plan Qty': row['Plan_Qty'],
            'Cycle Time (s)': row['Cycle_Time_s'],
            'Total Seconds': row['Production_Seconds'],
            'Total Hours': round(row['Production_Hours'], 2),
            'Type': 'Fixed'
        })
    
    # Flexible assignments
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Reference': part['Switch_Part_Reference'],
                'Plan Qty': part['Plan_Qty'],
                'Cycle Time (s)': part['Cycle_Time_s'],
                'Total Seconds': part['Production_Seconds'],
                'Total Hours': round(part['Production_Hours'], 2),
                'Type': 'Flexible'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Machine summary
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Utilization %'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Available Hours (3 days)'] = AVAILABLE_SECONDS_PER_MACHINE / 3600
    
    summary = summary[['Part Count', 'Total Hours', 'Available Hours (3 days)', 'Utilization %']]
    
    return plan_df, summary


def main():
    print("Processing production plan for 120-ton vertical machines...\n")
    
    agg_df = load_and_aggregate_data()
    fixed, flexible = calculate_loads(agg_df)
    
    flexible_assignments, machine_load_sec = assign_flexible_parts(flexible)
    
    plan_df, summary = create_final_plan(fixed, flexible_assignments)
    
    # Save to Excel
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_df.to_excel(writer, sheet_name='Detailed Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
    
    print("\n" + "="*70)
    print("         PRODUCTION PLAN GENERATED SUCCESSFULLY")
    print("="*70)
    
    print("\nMachine Summary (3-day horizon):")
    print(summary.round(2))
    
    print(f"\nDetailed plan saved to: {OUTPUT_FILE}")
    
    # Overload warning
    overloaded = summary[summary['Utilization %'] > 100]
    if not overloaded.empty:
        print("\nWARNING ─ OVERLOADED MACHINES:")
        print(overloaded)
    else:
        print("\nAll machines under 100% utilization → good balance.")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from collections import defaultdict

# ────────────────────────────────────────────────
#          CONFIGURATION - CHANGE THESE AS NEEDED
# ────────────────────────────────────────────────
FILE_PATH = r"C:\Your\Path\Here\your_file.xlsx"          # ← CHANGE THIS to your actual file path
SHEET_NAME = "Sheet1"                                    # ← change if your sheet name is different
OUTPUT_FILE = "production_plan_120ton_3days.xlsx"        # where results will be saved

# 120 tonnage vertical machines (excluding fixed-tool machines MP-11 & MP-15)
ALLOWED_MACHINES = ['MP-01', 'MP-05', 'MP-10', 'MP-17']

# Production capacity
HOURS_PER_DAY = 22
SECONDS_PER_DAY = HOURS_PER_DAY * 3600                   # 79,200 seconds/day
PLANNING_DAYS = 3
AVAILABLE_SECONDS_PER_MACHINE = SECONDS_PER_DAY * PLANNING_DAYS   # 237,600 seconds
AVAILABLE_HOURS_PER_MACHINE   = HOURS_PER_DAY * PLANNING_DAYS     # 66 hours

# Your column names (adjust only if they differ exactly)
COL_CHILD       = "Child Part"
COL_SWITCH      = "Switch Part Number"
COL_DAILY_PLAN  = "Daily Plan (of Switch Part Number)"
COL_MONTHLY_REQ = "Monthly Requirement (Child part)"
COL_MIN_QTY     = "Minimum Quantity (Child Part)"
COL_PLAN_QTY    = "Plan(Child Part calculated as Monthly Requirement / 31 * 3)"
COL_CYCLE_TIME  = "Cycle Time(of child part)"
COL_MACHINE     = "Vertical Machines"
# ────────────────────────────────────────────────

def load_and_aggregate_data():
    if FILE_PATH.lower().endswith('.csv'):
        df = pd.read_csv(FILE_PATH)
    else:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    
    df.columns = df.columns.str.strip()
    
    keep_cols = [COL_CHILD, COL_SWITCH, COL_DAILY_PLAN, COL_MONTHLY_REQ,
                 COL_MIN_QTY, COL_PLAN_QTY, COL_CYCLE_TIME, COL_MACHINE]
    df = df[keep_cols].copy()
    
    numeric_cols = [COL_PLAN_QTY, COL_CYCLE_TIME, COL_MONTHLY_REQ, COL_MIN_QTY, COL_DAILY_PLAN]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    df = df.dropna(subset=[COL_CHILD, COL_PLAN_QTY, COL_CYCLE_TIME])
    
    # Take FIRST occurrence of each Child Part only (no summing)
    df = df.sort_index()  # preserve original file order
    aggregated = df.groupby(COL_CHILD, as_index=False).first()
    
    # Show duplication info
    counts = df[COL_CHILD].value_counts().reset_index(name='Appearance_Count')
    aggregated = aggregated.merge(counts, on=COL_CHILD, how='left')
    
    aggregated = aggregated.rename(columns={
        COL_PLAN_QTY:    'Plan_Qty',
        COL_CYCLE_TIME:  'Cycle_Time_s',
        COL_MACHINE:     'Assigned_Machine',
        COL_SWITCH:      'Switch_Reference',
        COL_MONTHLY_REQ: 'Monthly_Req',
        COL_MIN_QTY:     'Min_Qty'
    })
    
    print(f"Original rows in file     : {len(df)}")
    print(f"Unique Child Parts        : {len(aggregated)}")
    print(f"Parts duplicated (≥2x)    : {len(aggregated[aggregated['Appearance_Count'] > 1])}")
    
    return aggregated


def calculate_loads(agg_df):
    agg_df['Production_Seconds'] = agg_df['Plan_Qty'] * agg_df['Cycle_Time_s']
    agg_df['Production_Hours']   = agg_df['Production_Seconds'] / 3600
    
    agg_df['Assigned_Machine'] = agg_df['Assigned_Machine'].astype(str).str.strip().str.upper()
    
    fixed = agg_df[agg_df['Assigned_Machine'].isin(ALLOWED_MACHINES)].copy()
    flexible = agg_df[~agg_df['Assigned_Machine'].isin(ALLOWED_MACHINES)].copy()
    
    print(f"Fixed-assigned parts      : {len(fixed)}")
    print(f"Flexible parts to assign  : {len(flexible)}")
    
    return fixed, flexible


def assign_flexible_parts(flexible):
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}  # current load in seconds
    assignments = defaultdict(list)
    
    # Sort largest jobs first → better balance
    flexible = flexible.sort_values('Production_Seconds', ascending=False)
    
    for _, row in flexible.iterrows():
        best_machine = min(machine_load, key=machine_load.get)
        assignments[best_machine].append(row)
        machine_load[best_machine] += row['Production_Seconds']
    
    return assignments, machine_load


def create_final_plan(fixed, flexible_assignments):
    plan_rows = []
    
    # Fixed
    for _, row in fixed.iterrows():
        plan_rows.append({
            'Machine': row['Assigned_Machine'],
            'Child Part': row[COL_CHILD],
            'Switch Reference': row['Switch_Reference'],
            'Plan Qty (3 days)': row['Plan_Qty'],
            'Cycle Time (s)': row['Cycle_Time_s'],
            'Total Seconds': int(row['Production_Seconds']),
            'Total Hours': round(row['Production_Hours'], 2),
            'Assignment Type': 'Fixed'
        })
    
    # Flexible
    for machine, parts in flexible_assignments.items():
        for part in parts:
            plan_rows.append({
                'Machine': machine,
                'Child Part': part[COL_CHILD],
                'Switch Reference': part['Switch_Reference'],
                'Plan Qty (3 days)': part['Plan_Qty'],
                'Cycle Time (s)': part['Cycle_Time_s'],
                'Total Seconds': int(part['Production_Seconds']),
                'Total Hours': round(part['Production_Hours'], 2),
                'Assignment Type': 'Flexible (balanced)'
            })
    
    plan_df = pd.DataFrame(plan_rows)
    
    # Summary per machine (3-day view)
    summary = plan_df.groupby('Machine').agg({
        'Total Seconds': 'sum',
        'Total Hours': 'sum',
        'Child Part': 'count'
    }).rename(columns={'Child Part': 'Part Count'})
    
    summary['Total Hours (3 days)'] = summary['Total Hours'].round(2)
    summary['Available Hours (3 days)'] = AVAILABLE_HOURS_PER_MACHINE
    summary['Utilization % (3 days)'] = (summary['Total Seconds'] / AVAILABLE_SECONDS_PER_MACHINE * 100).round(1)
    summary['Load Status'] = summary['Utilization % (3 days)'].apply(
        lambda x: 'Overloaded (>100%)' if x > 100 else 
                  'High (85-100%)'     if x > 85 else 
                  'Good (60-85%)'      if x > 60 else 'Low / Underutilized'
    )
    
    summary = summary[['Part Count', 'Total Hours (3 days)', 'Available Hours (3 days)', 
                       'Utilization % (3 days)', 'Load Status']]
    
    return plan_df, summary


def main():
    print("=== 120 Ton Vertical Machines - 3 Day Production Plan ===\n")
    print(f"Capacity per machine: {HOURS_PER_DAY} hrs/day × {PLANNING_DAYS} days = {AVAILABLE_HOURS_PER_MACHINE} hrs\n")
    
    agg_df = load_and_aggregate_data()
    fixed, flexible = calculate_loads(agg_df)
    
    flexible_assignments, machine_load_sec = assign_flexible_parts(flexible)
    
    plan_df, summary = create_final_plan(fixed, flexible_assignments)
    
    # Save results
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        plan_df.to_excel(writer, sheet_name='Detailed Plan', index=False)
        summary.to_excel(writer, sheet_name='Machine Summary')
    
    print("\n" + "="*75)
    print("              PRODUCTION PLAN GENERATED")
    print("="*75)
    
    print("\nMachine Summary (3-day plan, 22 productive hrs/day):")
    print(summary)
    
    print(f"\nDetailed assignment saved to: {OUTPUT_FILE}")
    
    overloaded = summary[summary['Utilization % (3 days)'] > 100]
    if not overloaded.empty:
        print("\nWARNING: Following machines are overloaded:")
        print(overloaded)
    else:
        print("\nNo overload detected. Plan is feasible within capacity.")
    
    # Optional: Daily average suggestion
    print("\nApproximate daily target per part (even split over 3 days):")
    daily_view = plan_df[['Machine', 'Child Part', 'Plan Qty (3 days)', 'Total Hours']].copy()
    daily_view['Daily Qty ≈'] = (daily_view['Plan Qty (3 days)'] / 3).round(0).astype(int)
    daily_view['Daily Hours ≈'] = (daily_view['Total Hours'] / 3).round(2)
    print(daily_view.sort_values(['Machine', 'Daily Hours ≈'], ascending=[True, False]))


if __name__ == "__main__":
    main()